In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import polars as pl
import torch
from torch.utils.data import DataLoader, TensorDataset

from fart.model.device import get_device
from fart.model.nbeats import NBeatsNet
from fart.model.nbeats_config import NBeatsConfig
from fart.model.nbeats_dataset import build_return_windows
from fart.model.train_model import prepare_training_data
from fart.utils import get_project_root
from fart.visualization.confidence_calibration import plot_confidence_calibration
from fart.visualization.plot_styles import apply_plot_styles

apply_plot_styles()

In [ ]:
assets_dir = get_project_root() / "assets"
market, interval = "BTC-EUR", "1d"

config = NBeatsConfig()
lookback = config.lookback

X_train, X_test, y_train, y_test = prepare_training_data(
    data_dir=assets_dir,
    market=market,
    interval=interval,
    months=None,
)

n_train = y_train.shape[0]
close_prices = pl.concat([y_train, y_test])
X_all, y_all = build_return_windows(close_prices, lookback)

n_train_windows = max(0, n_train - lookback - 1)
X_train_windows, y_train_windows = X_all[:n_train_windows], y_all[:n_train_windows]
X_test_windows, y_test_windows = X_all[n_train_windows:], y_all[n_train_windows:]

len(close_prices), X_train_windows.shape, X_test_windows.shape

In [ ]:
device = get_device()
num_members = 5

member_predictions: list[torch.Tensor] = []

for seed in range(num_members):
    torch.manual_seed(seed)
    model = NBeatsNet(config).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)

    train_loader = DataLoader(
        TensorDataset(X_train_windows, y_train_windows),
        batch_size=config.batch_size,
        shuffle=True,
    )

    model.train()
    epoch_loss = 0.0
    for epoch in range(config.epochs):
        epoch_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            mu, _log_sigma = model(X_batch).unbind(-1)
            loss = torch.nn.functional.mse_loss(mu, y_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * X_batch.shape[0]

    model.eval()
    with torch.no_grad():
        mu_test, _log_sigma_test = model(X_test_windows.to(device)).unbind(-1)
    member_predictions.append(mu_test.cpu())

    print(f"member {seed}: final epoch loss = {epoch_loss / len(X_train_windows):.6f}")

predictions = torch.stack(member_predictions)
predictions.shape

In [ ]:
mu_ensemble = predictions.mean(dim=0).numpy()
sigma_ensemble = predictions.std(dim=0).numpy()

error = np.abs(y_test_windows.numpy() - mu_ensemble)
confidence = 1 / (1 + sigma_ensemble)

print(f"confidence range: [{confidence.min():.5f}, {confidence.max():.5f}]")
print(f"error range: [{error.min():.6f}, {error.max():.6f}]")

In [ ]:
plot_confidence_calibration(confidence, error)

## Comparison to notebook 2.0

| | Self-reported confidence (notebook 2.0) | Ensemble disagreement (this notebook) |
|---|---|---|
| n (test windows) | 535 | 535 |
| Pearson r | 0.043 | -0.202 |
| Confidence range | [0.965, 0.976] | [0.94495, 0.99776] |

The ensemble-disagreement confidence shows a substantially stronger (though still weak-to-moderate) correlation with actual error than the self-reported confidence — |r| rose from 0.043 to 0.202, a roughly 4.7x increase — and its spread widened markedly, from a collapsed 0.011-wide band ([0.965, 0.976]) to a ~0.053-wide range ([0.94495, 0.99776]), meaning the ensemble no longer collapses to a near-constant value and both moves point toward ensemble disagreement being a more informative (if still imperfect) calibration signal than self-reported confidence.